# Nav Fun Training Model

Train a small open model on a playful message-classification task, all on an Apple Silicon Mac. The model learns to pick one of three labels: `greeting`, `question`, or `request`.

This notebook uses a tiny, fictional dataset to demonstrate the workflow. Results on these examples do not establish general-purpose accuracy. Run the cells in order. The model download and local training happen when you run the relevant cells.


In [ ]:
from pathlib import Path
import json, subprocess, sys, time

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'train.jsonl').exists(): ROOT = ROOT.parent
assert (ROOT / 'data' / 'train.jsonl').exists(), 'Start Jupyter from the repository directory.'
MODEL = 'mlx-community/Qwen3.5-0.8B-4bit'
DATA = ROOT / 'data'
train = [json.loads(x) for x in (DATA / 'train.jsonl').read_text().splitlines() if x]
valid = [json.loads(x) for x in (DATA / 'valid.jsonl').read_text().splitlines() if x]
print(f'{len(train)} training examples, {len(valid)} validation examples')
print(json.dumps(train[0], indent=2))


## 1. Try the base model

The first run downloads the model. We compare it on nine validation examples before and after fine-tuning. These examples are also used to monitor training, so this is not an independent test.


In [ ]:
from mlx_lm import load, generate
LABELS = {'greeting', 'question', 'request'}

def evaluate(model, tokenizer):
    rows = []
    for ex in valid:
        prompt = tokenizer.apply_chat_template(ex['messages'][:-1], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        start = time.perf_counter()
        raw = generate(model, tokenizer, prompt=prompt, max_tokens=16, verbose=False).strip()
        rows.append({'message': ex['messages'][1]['content'], 'expected': ex['messages'][-1]['content'], 'output': raw, 'seconds': round(time.perf_counter()-start, 3)})
    return rows

def show(rows):
    print(f"Exact matches: {sum(x['output'] == x['expected'] for x in rows)}/{len(rows)}")
    for row in rows: print(json.dumps(row))

model, tokenizer = load(MODEL)
base = evaluate(model, tokenizer)
show(base)


## 2. Train a small adapter

LoRA trains a small set of additional weights while leaving the base model intact. Each run uses a new timestamped directory under `adapters/`. The run time depends on your Mac and software versions.


In [ ]:
# Free the inference model before starting the trainer.
del model, tokenizer
import gc
gc.collect()
adapter = ROOT / 'adapters' / ('fun-' + time.strftime('%Y%m%d-%H%M%S'))
if adapter.exists(): raise FileExistsError(adapter)
cmd = [sys.executable, '-m', 'mlx_lm', 'lora', '--model', MODEL,
       '--train', '--mask-prompt', '--data', str(DATA), '--adapter-path', str(adapter),
       '--iters', '30', '--batch-size', '1', '--max-seq-length', '256',
       '--val-batches', '3', '--steps-per-report', '5', '--steps-per-eval', '30']
print('Adapter:', adapter, flush=True)
start = time.perf_counter()
result = subprocess.run(cmd, cwd=ROOT)
print(f'Elapsed: {time.perf_counter() - start:.1f} seconds; exit code: {result.returncode}')
if result.returncode != 0 or not (adapter / 'adapters.safetensors').exists():
    raise RuntimeError('Training did not produce an adapter. See the output above.')


## 3. Compare after training

Look at the individual outputs as well as the count. A tiny validation set can be misleading; for a serious test, use a larger independently authored set with a clear scoring policy.


In [ ]:
model, tokenizer = load(MODEL, adapter_path=str(adapter))
tuned = evaluate(model, tokenizer)
print('BASE')
show(base)
print('\nTUNED')
show(tuned)


## 4. Try your own messages

Edit the messages below and rerun the cell. Unexpected wording may produce unexpected labels.


In [ ]:
def classify(message):
    messages = [train[0]['messages'][0], {'role': 'user', 'content': message}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    return generate(model, tokenizer, prompt=prompt, max_tokens=16, verbose=False).strip()

for message in ['hey, thanks!', 'what is a planet?', 'suggest a game for two people']:
    print(message, '→', classify(message))
